In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

print("🥇 AGRÉGATIONS AVANCÉES SILVER → GOLD")
print("=" * 60)

df_silver = spark.read.table("water_catalog.silver.water_quality_clean")

print(f"📥 Lignes Silver : {df_silver.count():,}")


In [0]:
df_feat = (
    df_silver
    # Ratio sécurité
       .withColumn(
        "ratio_limite",
        F.when(
            F.col("limite_qualite") > 0,
            F.col("resultat_numerique") / F.col("limite_qualite")
        ).otherwise(0)
    )

    # Dépassement
    .withColumn(
        "depassement",
        F.when(F.col("ratio_limite") > 1, 1).otherwise(0)
    )

    # Gravité
    .withColumn(
        "niveau_depassement",
        F.when(F.col("ratio_limite") >= 5, "CRITIQUE")
         .when(F.col("ratio_limite") >= 2, "GRAVE")
         .when(F.col("ratio_limite") >= 1.2, "MODERE")
         .when(F.col("ratio_limite") >= 1, "LEGER")
         .otherwise("OK")
    )

    # Score ligne (pondération)
    .withColumn(
        "score_ligne",
        F.when(F.col("depassement") == 0, 1)
         .when(F.col("niveau_depassement") == "LEGER", 0.7)
         .when(F.col("niveau_depassement") == "MODERE", 0.4)
         .when(F.col("niveau_depassement") == "GRAVE", 0.2)
         .otherwise(0)
    )
)

df_feat.display()

In [0]:
dim_temps = (
    df_silver
    .select(
        "date_prelevement",
        "annee_prelevement",
        "mois_prelevement",
        "trimestre_prelevement"
    )
    .dropDuplicates()
)

display(dim_temps)

In [0]:
dim_commune = (
    df_silver
    .select(
        "code_commune",
        "nom_commune",
        "code_departement",
        "nom_departement",
        "nom_uge",
        "nom_distributeur",
        "nom_moa"
    )
    .dropDuplicates()
)
display(dim_commune)

In [0]:
dim_parametre = (
    df_silver
    .select(
        "code_parametre",
        "code_parametre_se",
        "code_parametre_cas",
        "libelle_parametre",
        "libelle_parametre_maj",
        "code_type_parametre",
        "libelle_unite",
        "code_unite"
    )
    .dropDuplicates()
)
display(dim_parametre)

In [0]:
dim_installation = (
    df_silver
    .select(
        "code_installation_amont",
        "nom_installation_amont",
        "code_lieu_analyse"
    )
    .dropDuplicates()
)
display(dim_installation)

In [0]:
fact_mesures = (
    df_silver
    .select(
        "code_prelevement",
        "date_prelevement",

        "code_commune",
        "code_departement",
        "code_parametre",

        "resultat_numerique",
        "resultat_alphanumerique",

        "limite_qualite",
        "reference_qualite",

        "conclusion_conformite_prelevement",
        "conformite_limites_bact_prelevement",
        "conformite_limites_pc_prelevement",

        "ingestion_date"
    )
)

display(fact_mesures)

In [0]:
from pyspark.sql import functions as F

df_feat = (
    df_silver

    # ratio qualité
    .withColumn(
        "ratio_limite",
        F.when(F.col("limite_qualite") > 0,
               F.col("resultat_numerique") / F.col("limite_qualite")).otherwise(0)
    )

    # dépassement
    .withColumn(
        "depassement",
        F.when(F.col("ratio_limite") > 1, 1).otherwise(0)
    )

    # niveau risque
    .withColumn(
        "niveau_risque",
        F.when(F.col("ratio_limite") >= 5, "CRITIQUE")
         .when(F.col("ratio_limite") >= 2, "GRAVE")
         .when(F.col("ratio_limite") >= 1, "MODERE")
         .otherwise("OK")
    )
)
display(df_feat)

In [0]:
gold_commune_kpi = (
    df_feat
    .groupBy(
        "code_commune",
        "nom_commune",
        "code_departement",
        "nom_departement"
    )
    .agg(
        F.count("*").alias("nb_mesures"),
        F.countDistinct("code_parametre").alias("nb_parametres"),

        F.avg("resultat_numerique").alias("moyenne_valeur"),

        F.sum("depassement").alias("nb_depassements"),

        F.round(
            F.sum("depassement") / F.count("*") * 100, 2
        ).alias("taux_depassement_pct"),

        F.max("ratio_limite").alias("max_ratio")
    )

    .withColumn(
        "score_qualite",
        100 - F.col("taux_depassement_pct")
    )

    .withColumn(
        "classe_qualite",
        F.when(F.col("score_qualite") >= 98, "EXCELLENT")
         .when(F.col("score_qualite") >= 95, "BON")
         .when(F.col("score_qualite") >= 90, "MOYEN")
         .otherwise("MAUVAIS")
    )
)
display(gold_commune_kpi)

In [0]:
spark.sql("DROP TABLE IF EXISTS water_catalog.gold.dim_commune")
spark.sql("DROP TABLE IF EXISTS water_catalog.gold.dim_parametre")
dim_commune.write.mode("overwrite").saveAsTable("water_catalog.gold.dim_commune")
dim_temps.write.mode("overwrite").saveAsTable("water_catalog.gold.dim_temps")
dim_parametre.write.mode("overwrite").saveAsTable("water_catalog.gold.dim_parametre")
dim_installation.write.mode("overwrite").saveAsTable("water_catalog.gold.dim_installation")
fact_mesures.write.mode("overwrite").saveAsTable("water_catalog.gold.fact_mesures")
gold_commune_kpi.write.mode("overwrite").saveAsTable("water_catalog.gold.commune_kpi")
#gold_anomalies.write.mode("overwrite").saveAsTable("water_catalog.gold.anomalies")
#gold_ranking.write.mode("overwrite").saveAsTable("water_catalog.gold.ranking")

print("✅ GOLD tables créées")

In [0]:
spark.range(10).show()